Project: /meridian/_project.yaml
Book: /meridian/_book.yaml

<style>
devsite-code .tfo-notebook-code-cell-output {
  max-height: 300px;
  overflow: auto;
  background: rgba(255, 247, 237, 1);  /* orange bg to distinguish from input code cells */
}

devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
  background: rgba(255, 247, 237, .7);  /* orange bg to distinguish from input code cells */
}

devsite-code[dark-code] .tfo-notebook-code-cell-output {
  background: rgba(64, 78, 103, 1);  /* medium slate */
}

devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
  background: rgba(64, 78, 103, .7);  /* medium slate */
}

/* override default table styles for notebook buttons */
.devsite-table-wrapper .tfo-notebook-buttons {
  display: inline-block;
  margin-left: 3px;
  width: auto;
}

.tfo-notebook-buttons tr {
  background: 0;
  border: 0;
}

.tfo-notebook-buttons td {
  padding-left: 0;
  padding-right: 20px;
}

.tfo-notebook-buttons {
  --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
}

.tfo-notebook-buttons a,
.tfo-notebook-buttons :link,
.tfo-notebook-buttons :visited {
  border-radius: 8px;
  box-shadow: var(--tfo-notebook-buttons-box-shadow);
  color: #202124;
  padding: 12px 24px;
  transition: box-shadow 0.2s;
}

.tfo-notebook-buttons a:hover,
.tfo-notebook-buttons a:focus {
  box-shadow: var(--tfo-notebook-buttons-box-shadow);
}

.tfo-notebook-buttons td > a {
  -webkit-box-align: center;
  -ms-flex-align: center;
  align-items: center;
  display: -webkit-box;
  display: -ms-flexbox;
  display: flex;
}

.tfo-notebook-buttons td > a > img {
  margin-right: 8px;
}
</style>

<table class="tfo-notebook-buttons tfo-api nocontent" align="left">
  <tbody>
    <tr>
      <td>
        <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
      </td>
      <td>
        <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
      </td>
    </tr>
  </tbody>
</table>

# **Introduction to Meridian Demo**

Welcome to the Meridian end-to-end demo. This simplified demo showcases the fundamental functionalities and basic usage of the library, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install and Enviroment Configuration</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#quality-checks">Run post-modeling quality checks</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
  <li><a href="#scenario-planning">Interactive Scenario Planning</a></li>
</ol>


Note that this notebook skips all of the exploratory data analysis and preprocessing steps. It assumes that you have completed these tasks before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

<a name="install"></a>
## Step 0: Install and Enviroment Configuration

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [3]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda,schema]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda,schema]==1.3.1

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda,schema] @ git+https://github.com/google/meridian.git@main"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.8/490.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 67.7 MB/s eta 0:00:00
  Attempting uninstall: natsort
    Found existing installation: natsort 8.4.0
    Uninstalling natsort-8.4.0:
      Successfully uninstalled natsort-8.4.0
  Attempting uninstall: arviz
    Found existing installation: arviz 0.22.0
    Uninstalling arviz-0.22.0:
      Successfully uninstalled arviz-0.22.0


In [24]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 13.6 gigabytes of available RAM

Num GPUs Available:  1
Num CPUs Available:  1


3\. Mount a storage. Use `meridian_root` to refer the mounted root. The mounted root will be used to <a href="#save-model">save trained model</a>, <a href="#generate-summary">stage two-pager output</a> and <a href="#scenario-planning">generate scenario planning dashboard</a>.

For Colab Free/Pro user, we will use the `MyDrive` folder in Google Drive as the external storage. For Colab Enterprise user, we will use <a href="https://docs.cloud.google.com/storage/docs/cloud-storage-fuse/overview">Cloud FUSE</a> to mount a GCS bucket.

In [5]:
# @markdown If you are using Colab Free, Colab Pro, run this cell to mount your Google Drive.
from google.colab import drive
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

Mounted at /content/drive


In [ ]:
# @markdown If you are using Colab Enterprise, uncomment and run this cell to mount a GCS bucket.
# import os

# project_id = ""# @param {"type":"string","placeholder": "Cloud project id"}
# bucket_name = "" # @param {"type":"string","placeholder": "GCS bucket that contains your model"}
# subfolder = "" # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# os.environ['GOOGLE_CLOUD_PROJECT'] = project_id
# !gcloud config set project {project_id}
# !gcloud auth login
# !mkdir /content/{bucket_name}

# # Uncomment below if you don't have gcsfuse installed
# #!echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
# #!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
# #!apt-get update
# #!apt-get install gcsfuse

# !gcsfuse --implicit-dirs {bucket_name} /content/{bucket_name}
# meridian_root = f'/content/{bucket_name}/{subfolder}'
# is_enterprise_user=True
#!fusermount -u /content/{bucket_name}

<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv) as follows.

1\. Read the data into a Pandas DataFrame.

In [6]:
df = pd.read_csv(
  "https://raw.githubusercontent.com/pstat197/BlueAlpha3-Synergy-Analysis/refs/heads/meridian_modeling/data/monthly_mocha.csv"
)

In [7]:
df = df.drop(columns=["date"], errors="ignore")
df = df.loc[:, (df != 0).any(axis=0)]

print(df.head())

   subscriptions   meta_spend  meta_impressions  google_spend  \
0          15540  91538.06648          16572258   116667.9945   
1          14525  93840.18612          25300600   180486.9558   
2          16880  48403.06780          14099214   200817.3250   
3          20113  49470.96783          13652072   215770.9242   
4          16492  48948.28744          10121002   209231.9668   

   google_impressions  snapchat_spend  snapchat_impressions  tiktok_spend  \
0             6473132     94750.04035               3420454           0.0   
1             9487127     99447.23218               3235285           0.0   
2             7909118     84738.57435               4766750           0.0   
3             7789279     83204.40500               4022680           0.0   
4             6806878     82642.37271               4532105           0.0   

   tiktok_impressions  moloco_spend  moloco_impressions  liveintent_spend  \
0                   0   6564.524233              367206       37766.4

In [8]:
# Prepare for Granger

# Keep only spend columns (remove impressions)
spend_cols = [c for c in df.columns if "_spend" in c]

# Create dataset for Granger (exclude target)
df_gc = df[spend_cols].copy()

# Difference data to make it stationary
df_gc_diff = df_gc.diff().dropna()

print("Columns used for Granger:", spend_cols)

Columns used for Granger: ['meta_spend', 'google_spend', 'snapchat_spend', 'tiktok_spend', 'moloco_spend', 'liveintent_spend', 'beehiiv_spend', 'amazon_spend']


In [9]:
print("Spend cols:", spend_cols)
print("Shape original:", df.shape)
print("Shape differenced:", df_gc_diff.shape)

Spend cols: ['meta_spend', 'google_spend', 'snapchat_spend', 'tiktok_spend', 'moloco_spend', 'liveintent_spend', 'beehiiv_spend', 'amazon_spend']
Shape original: (74, 17)
Shape differenced: (73, 8)


In [10]:
# Run Granger Causality
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations

gc_results = {}

channels = df_gc_diff.columns

# Loop through all ordered pairs (x1, x2)
# Test: does x2 Granger-cause x1?
for x1, x2 in permutations(channels, 2):

    test_result = grangercausalitytests(
        df_gc_diff[[x1, x2]],
        maxlag=3,
        verbose=False
    )

    # Extract p-values for each lag (SSR F-test)
    p_values = {
        lag: test_result[lag][0]['ssr_ftest'][1]
        for lag in range(1, 4)
    }

    gc_results[(x1, x2)] = p_values

# Convert results to DataFrame
gc_df = pd.DataFrame(gc_results).T
gc_df.columns = [f"lag_{i}_pvalue" for i in range(1, 4)]

print("\nRaw Granger Results:")
print(gc_df.head())

long_df = (
    gc_df.reset_index()
    .rename(columns={"level_0": "target", "level_1": "driver"})
    .melt(id_vars=["target", "driver"], var_name="lag", value_name="p_value")
)

# Extract numeric lag from string
long_df["lag"] = long_df["lag"].str.extract(r"lag_(\d+)_pvalue").astype(int)

summary_df = (
    long_df.sort_values(["target", "driver", "p_value"])
    .groupby(["target", "driver"], as_index=False)
    .first()
    .rename(columns={"p_value": "min_pvalue", "lag": "best_lag"})
)

m = len(summary_df)

summary_df["bonferroni_p"] = np.minimum(summary_df["min_pvalue"] * m, 1.0)
summary_df["sig_bonf_05"] = summary_df["bonferroni_p"] < 0.05

def fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    ranked = pvals[order]

    q = ranked * len(pvals) / (np.arange(1, len(pvals) + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    out = np.empty_like(q)
    out[order] = q
    return out

summary_df["fdr_q"] = fdr_bh(summary_df["min_pvalue"].values)
summary_df["sig_fdr_05"] = summary_df["fdr_q"] < 0.05

drivers_bonf = summary_df.loc[
    summary_df["sig_bonf_05"], "driver"
].unique().tolist()

# Balanced (recommended)
drivers_fdr = summary_df.loc[
    summary_df["sig_fdr_05"], "driver"
].unique().tolist()

print("\n==============================")
print("SIGNIFICANT DRIVERS (Bonferroni):")
print(drivers_bonf)

print("\nSIGNIFICANT DRIVERS (FDR):")
print(drivers_fdr)

print("\n==============================")
print("TOP RELATIONSHIPS:")
print(
    summary_df
    .sort_values(["bonferroni_p", "fdr_q"])
    .head(15)[
        ["target", "driver", "best_lag", "min_pvalue",
         "bonferroni_p", "fdr_q", "sig_bonf_05", "sig_fdr_05"]
    ]
)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print resul


Raw Granger Results:
                             lag_1_pvalue  lag_2_pvalue  lag_3_pvalue
meta_spend google_spend          0.373219      0.844775      0.867218
           snapchat_spend        0.556460      0.853165      0.942809
           tiktok_spend          0.075610      0.110496      0.212783
           moloco_spend          0.138498      0.479829      0.583483
           liveintent_spend      0.297042      0.342096      0.542898

SIGNIFICANT DRIVERS (Bonferroni):
['meta_spend', 'liveintent_spend']

SIGNIFICANT DRIVERS (FDR):
['meta_spend', 'liveintent_spend']

TOP RELATIONSHIPS:
              target            driver  best_lag    min_pvalue  bonferroni_p  \
3       amazon_spend        meta_spend         3  8.202508e-09  4.593404e-07   
16      google_spend  liveintent_spend         3  5.795737e-04  3.245613e-02   
11     beehiiv_spend      moloco_spend         2  5.869067e-03  3.286677e-01   
28        meta_spend      amazon_spend         1  1.301752e-02  7.289810e-01   
47   

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print resul

In [11]:
# Create Lagged Features for MMM
df_lagged = df.copy()

# Create lag features for each spend channel
for col in spend_cols:
    for lag in range(1, 3):  # lag 1 and 2
        df_lagged[f"{col}_lag{lag}"] = df_lagged[col].shift(lag)

# Drop NA from lagging
df_lagged = df_lagged.dropna().reset_index(drop=True)

print(df_lagged.head())

   subscriptions   meta_spend  meta_impressions  google_spend  \
0          16880  48403.06780          14099214   200817.3250   
1          20113  49470.96783          13652072   215770.9242   
2          16492  48948.28744          10121002   209231.9668   
3          15459  47241.55544           8953395   197595.7996   
4          15798      0.00000                 0   203264.8346   

   google_impressions  snapchat_spend  snapchat_impressions  tiktok_spend  \
0             7909118     84738.57435               4766750           0.0   
1             7789279     83204.40500               4022680           0.0   
2             6806878     82642.37271               4532105           0.0   
3            11115242     87738.66067               3995130           0.0   
4             9827238     90905.70724               4935027           0.0   

   tiktok_impressions  moloco_spend  ...  tiktok_spend_lag1  \
0                   0   9714.794608  ...                0.0   
1                   

In [12]:
# Meridian REQUIRES these columns

# Time must be string format YYYY-MM-DD
df_lagged["time"] = pd.date_range(
    start="2020-01-01",
    periods=len(df_lagged),
    freq="D"
).astype(str)

# Required placeholders
df_lagged["revenue_per_conversion"] = 1.0
df_lagged["population"] = 1.0

print(df_lagged[["time"]].head())

         time
0  2020-01-01
1  2020-01-02
2  2020-01-03
3  2020-01-04
4  2020-01-05


In [13]:
# Get all spend columns
media_spend_cols = [c for c in df_lagged.columns if "_spend" in c]

# Build matching impression columns
media_cols = []
final_spend_cols = []
channels = []

for spend_col in media_spend_cols:
    base = spend_col.replace("_spend", "")
    impression_col = f"{base}_impressions"

    # Only keep if BOTH exist
    if impression_col in df_lagged.columns:
        final_spend_cols.append(spend_col)
        media_cols.append(impression_col)
        channels.append(base)

# Overwrite with filtered lists
media_spend_cols = final_spend_cols

print("Channels used:", channels)
print("Spend cols:", media_spend_cols)
print("Impression cols:", media_cols)

Channels used: ['meta', 'google', 'snapchat', 'tiktok', 'moloco', 'liveintent', 'beehiiv', 'amazon']
Spend cols: ['meta_spend', 'google_spend', 'snapchat_spend', 'tiktok_spend', 'moloco_spend', 'liveintent_spend', 'beehiiv_spend', 'amazon_spend']
Impression cols: ['meta_impressions', 'google_impressions', 'snapchat_impressions', 'tiktok_impressions', 'moloco_impressions', 'liveintent_impressions', 'beehiiv_impressions', 'amazon_impressions']


In [14]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column="subscriptions",
    default_revenue_per_kpi_column="revenue_per_conversion",
)

builder = (
    builder.with_kpi(df_lagged)
    .with_revenue_per_kpi(df_lagged)
    .with_population(df_lagged)
    .with_media(
        df_lagged,
        media_cols=media_spend_cols, # Changed to use spend columns as media volume
        media_spend_cols=media_spend_cols,
        media_channels=channels
    )
)

# IMPORTANT: don't call this "data"
mmm_data = builder.build()

print(type(mmm_data))

<class 'meridian.data.input_data.InputData'>


/usr/local/lib/python3.12/dist-packages/meridian/data/input_data_builder.py:715: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(


In [15]:
# Initialize Model
prior = spec.prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=0.2,
        scale=0.9,
        name=constants.ROI_M
    )
)

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True
)

mmm = model.Meridian(
    input_data=mmm_data,
    model_spec=model_spec
)

print("Model ready")

Model ready


/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:74: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [29]:
# Bayesian sampling (this is where learning happens)

# Explicitly sample the prior to make it available for summarization
mmm.sample_prior(n_draws=500)

mmm.sample_posterior(
    n_chains=1,
    n_adapt=500,
    n_burnin=500,
    n_keep=500
)

print("Sampling complete")

Sampling complete


/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


In [17]:
analyzer_obj = analyzer.Analyzer(mmm)

# Extract key outputs
roi = analyzer_obj.roi()
mroi = analyzer_obj.marginal_roi()
inc = analyzer_obj.incremental_outcome()

print("ROI:\n", roi)
print("\nMarginal ROI:\n", mroi)
print("\nIncremental Outcome:\n", inc)

/tmp/ipykernel_5740/250213931.py:1: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  analyzer_obj = analyzer.Analyzer(mmm)


ROI:
 tf.Tensor(
[[[0.01218197 0.0469093  0.06052993 ... 0.28384176 0.2052525  0.2572193 ]
  [0.05394495 0.02870726 0.10198595 ... 0.214532   0.32942718 0.14725192]
  [0.02659157 0.01924864 0.12490469 ... 0.14677416 0.15638034 0.5090143 ]
  ...
  [0.04278279 0.02082794 0.08753571 ... 0.15468282 0.3879808  0.15975668]
  [0.01986199 0.02623131 0.1133446  ... 0.14788006 0.18949966 0.46603176]
  [0.03635851 0.01724927 0.0859254  ... 0.17107634 0.18288516 0.07818639]]], shape=(1, 500, 8), dtype=float32)

Marginal ROI:
 tf.Tensor(
[[[0.00838873 0.00409068 0.0342683  ... 0.07174474 0.15120104 0.11131638]
  [0.02985199 0.00516085 0.05169813 ... 0.10076351 0.05325855 0.02616326]
  [0.00613965 0.00977739 0.08736236 ... 0.05575011 0.02382676 0.1215968 ]
  ...
  [0.02765541 0.00879128 0.04445937 ... 0.03916656 0.16016638 0.09200259]
  [0.01257982 0.00547924 0.07332025 ... 0.0479352  0.09663959 0.25102955]
  [0.01488334 0.00564218 0.03614786 ... 0.09655239 0.1082808  0.04839924]]], shape=(1, 500, 8

In [18]:
import numpy as np
import pandas as pd

analyzer_obj = analyzer.Analyzer(mmm)

roi = analyzer_obj.roi().numpy()
mroi = analyzer_obj.marginal_roi().numpy()
inc = analyzer_obj.incremental_outcome().numpy()

roi = np.squeeze(roi)
mroi = np.squeeze(mroi)
inc = np.squeeze(inc)

roi_mean = roi.mean(axis=0)
roi_p5 = np.percentile(roi, 5, axis=0)
roi_p95 = np.percentile(roi, 95, axis=0)

mroi_mean = mroi.mean(axis=0)
mroi_p5 = np.percentile(mroi, 5, axis=0)
mroi_p95 = np.percentile(mroi, 95, axis=0)

inc_mean = inc.mean(axis=0)

results = pd.DataFrame({
    "ROI_mean": roi_mean,
    "ROI_p5": roi_p5,
    "ROI_p95": roi_p95,
    "Marginal_ROI_mean": mroi_mean,
    "Incremental_mean": inc_mean
})

print(results)

/tmp/ipykernel_5740/392528934.py:4: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  analyzer_obj = analyzer.Analyzer(mmm)


   ROI_mean    ROI_p5   ROI_p95  Marginal_ROI_mean  Incremental_mean
0  0.027037  0.012462  0.048650           0.014909      17703.431641
1  0.023021  0.010421  0.038753           0.007489     230241.578125
2  0.081885  0.038617  0.131708           0.047758     339676.875000
3  0.040984  0.017377  0.085470           0.013759      71832.445312
4  0.086233  0.037181  0.152309           0.025326      70500.664062
5  0.175577  0.055559  0.347390           0.071604     126710.156250
6  0.219233  0.082167  0.425797           0.104826      63559.625000
7  0.220912  0.079925  0.424597           0.116396       3477.209473


In [32]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_5740/879150696.py:1: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/tensorflow_probability/python/mcmc/diagnostic.py:580: RuntimeWarning: divide by zero encountered in divide
  return (n / (n - 1.)) * biased_var
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Review,"The posterior probability that the baseline is negative is 0.53. This indicates that the baseline time series occasionally dips into negative values. We recommend visually inspecting the baseline time series in the Model Fit charts, but don't be overly concerned. An occasional, small dip may indicate minor statistical error, which is inherent in any model."
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 0.95. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Pass,"R-squared = 0.9171, MAPE = 0.0496, and wMAPE = 0.0466. These goodness-of-fit metrics are intended for guidance and relative comparison."
Prior-posterior shift,Pass 8/8 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.


<a name="model-diagnostics"></a>
## Step 4: Run model diagnostics

To further assess convergence and model fit, you can use the methods from `visualizer` module.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [20]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

/usr/local/lib/python3.12/dist-packages/tensorflow_probability/python/mcmc/diagnostic.py:580: RuntimeWarning: divide by zero encountered in divide
  return (n / (n - 1.)) * biased_var
/usr/local/lib/python3.12/dist-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


alt.LayerChart(...)

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [21]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


alt.LayerChart(...)

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 5: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [22]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [30]:
filepath = meridian_root
start_date = '2020-01-01'
end_date = '2020-03-12'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:3356: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


In [25]:
print('Max date in df_lagged:', df_lagged['time'].max())

Max date in df_lagged: 2020-03-12


Here is a preview of the two-page output based on the simulated data:

In [31]:
IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

Dataset,R-squared,MAPE,wMAPE
All Data,0.92,5%,5%


For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 6: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

Alternatively, if you would like to have a sharable interactive dashboard, check out [Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 32 s, sys: 1.64 s, total: 33.7 s
Wall time: 39.7 s


2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [ ]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Channel4,22%,28%
Channel3,40%,28%
Channel0,18%,24%
Channel1,14%,13%
Channel2,6%,7%


For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).


Optimization can also be performed on a hypothetical data representing a future scenario. The new data takes the same structure as the input data and encodes an anticipated flighting pattern, cost per media unit, and revenue per kpi.

3\. Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv) into Pandas DataFrame.

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv"
)



4\. New data is read from a csv file and converted into a set of multi-dimensional arrays. The arrays are used to construct a `DataTensors` instance, which is passed to `optimize()` as the `new_data` argument.

Constructing a `DataTensors` instance requires that all arrays have "time" and "geo" dimensions. Alternatively, the `BudgetOptimizer.create_optimization_tensors` method can be used to construct a `DataTensors` instance. This helper method can simplify the process, particularly when you do not need "time" and "geo" dimensions for all inputs. For example, it can be convenient if you want to assume a constant "revenue per kpi" or "cost per media unit" for all geos and time periods.

In [ ]:
n_geos = mmm.n_geos
n_media_channels = mmm.n_media_channels
n_non_media_channels = mmm.n_non_media_channels
n_organic_media_channels = mmm.n_organic_media_channels

# The number of time periods and time range do not need to match the input data.
df[constants.TIME] = pd.to_datetime(df[constants.TIME], errors='coerce')
unique_times = sorted(df[constants.TIME].unique())
n_times = len(unique_times)

geos = mmm.input_data.geo.values
media_channels = mmm.input_data.media_channel.values
media_cols = [f"{channel}_impression" for channel in media_channels]
media_spend_cols = [f"{channel}_spend" for channel in media_channels]
non_media_treatment_cols = ['Promo']
organic_media_cols = ['Organic_channel0_impression']
organic_media_channels = ['Organic_channel0']
revenue_per_kpi_col='revenue_per_conversion'
times_str = [time.strftime(constants.DATE_FORMAT) for time in unique_times]

media_np = np.zeros((n_geos, n_times, n_media_channels))
media_spend_np = np.zeros((n_geos, n_times, n_media_channels))
non_media_treatment_np = np.zeros((n_geos, n_times, n_non_media_channels))
organic_media_np = np.zeros((n_geos, n_times, n_organic_media_channels))
revenue_per_kpi_np = np.zeros((n_geos, n_times))

df_grouped = df.set_index([constants.GEO, constants.TIME])
for geo_idx, geo in enumerate(geos):
  for time_idx, time in enumerate(unique_times):
    row = df_grouped.loc[(geo, time)]
    media_np[geo_idx, time_idx, :] = row[media_cols].values
    media_spend_np[geo_idx, time_idx, :] = row[media_spend_cols].values
    non_media_treatment_np[geo_idx, time_idx, :] = row[non_media_treatment_cols].values
    organic_media_np[geo_idx, time_idx, :] = row[organic_media_cols].values
    revenue_per_kpi_np[geo_idx, time_idx] = row[revenue_per_kpi_col].item()

data_tensors = analyzer.DataTensors(
    media=tf.convert_to_tensor(media_np, dtype=tf.float32),
    media_spend=tf.convert_to_tensor(media_spend_np, dtype=tf.float32),
    non_media_treatments=tf.convert_to_tensor(non_media_treatment_np, dtype=tf.float32),
    organic_media=tf.convert_to_tensor(organic_media_np, dtype=tf.float32),
    revenue_per_kpi=tf.convert_to_tensor(revenue_per_kpi_np, dtype=tf.float32),
    time=tf.convert_to_tensor(times_str, dtype=tf.string),
)
# Default values for `budget` and `pct_of_spend` are derived from the `new_data`,
# but these values can be overridden without modifying the `new_data` itself.
hypothetical_optimization_results = budget_optimizer.optimize(
    new_data=data_tensors,
    budget=50_000_000,
    pct_of_spend=[.2, .1, .2, .2, .3]
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:397: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:397: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(


5\. Export the 2-page HTML optimization report.

In [ ]:
filepath = meridian_root
hypothetical_optimization_results.output_optimization_summary(
    'hypothetical_optimization_output.html', filepath
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:397: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:397: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(


In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/hypothetical_optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Channel4,30%,27%
Channel0,20%,25%
Channel3,20%,23%
Channel2,20%,14%
Channel1,10%,10%


<a name="save-model"></a>
## Step 7: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [ ]:
file_path = f'{meridian_root}/saved_mmm.binpb'
meridian_serde.save_meridian(mmm, file_path)
print(f'model is saved at {file_path}')

model is saved at /content/drive/MyDrive//saved_mmm.binpb


Run the following codes to load the saved model:

In [ ]:
mmm = meridian_serde.load_meridian(file_path)

/usr/local/lib/python3.12/dist-packages/meridian/schema/utils/proto_enum_converter.py:105: UserWarning: Paid media prior type is unspecified. Resolving to default: PAID_MEDIA_PRIOR_TYPE_UNSPECIFIED.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1538: UserWarning: The group trace is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/knots.py:505: RuntimeWarning: overflow encountered in cast
  np.float32(math.comb(ncol, design_mat.shape[1]))


In [ ]:
# @markdown If a GCS bucket is mounted, run this cell to unmount it.
!fusermount -u /content/{bucket_name}

<a name="scenario-planning"></a>
## Step 8: Interactive Scenario Planning
[Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner) is a tool that allow advertisers to create a sharable and interactive dashboard from Colab; you can reuse the saved model from this Colab for dashboard generation.